# DCG Colab Runner (T4)

This notebook is prepared for Google Colab with a T4 GPU runtime.

## Before running
1. In Colab, set Runtime -> Change runtime type -> T4 GPU.
2. Make sure the project folder is available at `/content/DCG` (clone or upload).
3. Ensure dataset files are present under `/content/DCG/data`.

In [ ]:
import os
import sys
import platform
import torch

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Optional: clone your repo
If DCG is not already available in `/content/DCG`, uncomment and use this cell.

In [ ]:
# REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'
# !git clone $REPO_URL /content/DCG

In [ ]:
!pip install -q numpy scipy scikit-learn munkres tqdm

In [ ]:
PROJECT_DIR = '/content/DCG'
assert os.path.isdir(PROJECT_DIR), f'Missing project folder: {PROJECT_DIR}'

required = [
    'configure.py',
    'datasets.py',
    'ICDM.py',
    'get_indicator_matrix_A.py',
    'data/synthetic3d.mat',
]

missing = [p for p in required if not os.path.exists(os.path.join(PROJECT_DIR, p))]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

%cd /content/DCG
if '/content/DCG' not in sys.path:
    sys.path.insert(0, '/content/DCG')

print('Project ready at', PROJECT_DIR)

## T4-aware smoke run
This mirrors your current smoke flow but uses GPU automatically when available.

In [ ]:
import itertools
import random
import numpy as np
import torch

from configure import get_default_config
from datasets import load_data
from get_indicator_matrix_A import get_mask
from ICDM import icdm

config = get_default_config('Synthetic3d')
config['dataset'] = 'Synthetic3d'
config['training']['epoch'] = 2
config['training']['batch_size'] = 64
config['noise_scheduler']['num_timesteps'] = 20
config['training']['lambda_mmi'] = 0.01
config['training']['lambda_cluster'] = 0.1
config['training']['lambda_hc'] = 0.1
config['training']['mmi_temperature'] = 1.0
config['training']['mmi_internal_lambda'] = 1.0
config['print_num'] = 1

seed = config['training']['seed']
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

x_list, y_list = load_data(config)
x1_train_raw = x_list[0]
x2_train_raw = x_list[1]

missing_rate = 0.3
mask = get_mask(2, x1_train_raw.shape[0], missing_rate)

x1_train = x1_train_raw * mask[:, 0][:, np.newaxis]
x2_train = x2_train_raw * mask[:, 1][:, np.newaxis]

x1_train = torch.from_numpy(x1_train).float().to(device)
x2_train = torch.from_numpy(x2_train).float().to(device)
mask = torch.from_numpy(mask).long().to(device)

model = icdm(config)
model.to_device(device)

optimizer = torch.optim.Adam(
    itertools.chain(
        model.autoencoder1.parameters(),
        model.autoencoder2.parameters(),
        model.df1.parameters(),
        model.df2.parameters(),
        model.clusterLayer.parameters(),
        model.AttentionLayer.parameters(),
    ),
    lr=config['training']['lr'],
)

acc, nmi, ari = model.train(config, x1_train, x2_train, y_list, mask, optimizer, device)
print('SMOKE_RESULT', f'ACC={acc:.4f}', f'NMI={nmi:.4f}', f'ARI={ari:.4f}')

## Optional: run your script directly
Use this if you prefer the script path over notebook-based execution.

In [ ]:
# !python smoke_test.py